# Práctica 2 · Paso 3 — Matriz de estrategias de mitigación y adaptación

**Cambio Climático · Ingeniería Ambiental · ESPAM MFL** · Material CC-IAM-P2-03 · Versión 1.0 · noviembre de 2026

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/crispaliz/cambio-climatico-espam/blob/main/practicas/ambiental/practica-2/IA2_03_matriz_estrategias.ipynb)

**Objetivo.** Proponer y priorizar estrategias para su provincia que respondan a **lo que más emite** (paso 1, inventario) y a **lo que más riesgo genera** en el sitio evaluado (paso 2, ficha), con una evaluación multicriterio transparente.

**Producto.** Matriz de opciones por criterios, ranking ponderado, prueba de robustez de los pesos, gráfico de impacto frente a factibilidad y cartera priorizada con sinergias y alertas de mala adaptación. Tercera pieza del capítulo 2 y evidencia de la **evaluación formativa**.

**Requisitos.** Haber ejecutado `IA2_01_inventario_emisiones` e `IA2_02_ficha_riesgo` con la misma provincia.

**Cómo trabajar.** En Colab: **Archivo › Guardar una copia en Drive**. Ejecute hasta la celda 4: le sugiere opciones del catálogo según sus resultados y le imprime un bloque para copiar. Péguelo en la celda 6, quédese con las que elija, califíquelas de 1 a 5 **con su justificación** y ejecute el resto. Puede agregar opciones propias en la celda 5.

## 1. Datos del estudiante

In [ ]:
# @title Datos del estudiante (la misma provincia de los pasos 1 y 2)
ESTUDIANTE = "Nombre Apellido"  # @param {type:"string"}
PROVINCIA = "Manabí"  # @param ["Azuay", "Bolívar", "Cañar", "Carchi", "Chimborazo", "Cotopaxi", "El Oro", "Esmeraldas", "Galápagos", "Guayas", "Imbabura", "Loja", "Los Ríos", "Manabí", "Morona Santiago", "Napo", "Orellana", "Pastaza", "Pichincha", "Santa Elena", "Santo Domingo de los Tsáchilas", "Sucumbíos", "Tungurahua", "Zamora Chinchipe"]

## 2. Entorno e insumos de los pasos 1 y 2

In [ ]:
import os, sys, pathlib, subprocess, datetime, json
REPO_URL = "https://github.com/crispaliz/cambio-climatico-espam"
REPO_DIR = pathlib.Path(os.environ.get("CC_REPO", "/content/cambio-climatico-espam"))
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "-q"], check=False, stderr=subprocess.DEVNULL)
sys.path.insert(0, str(REPO_DIR))

from herramientas import estrategias as ES, riesgo as RG
from herramientas.limites import slug

try:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = pathlib.Path("/content/drive/MyDrive")
except Exception:
    BASE = pathlib.Path(os.environ.get("CC_BASE", "."))

import numpy as np, pandas as pd, matplotlib.pyplot as plt
SP = slug(PROVINCIA)
RES = BASE / "CC_Ambiental" / "practica-2" / SP
necesarios = [RES / f"IA2_01_categorias_principales_{SP}.csv", RES / "IA2_01_metadatos.json", RES / f"IA2_02_riesgo_{SP}.csv", RES / "IA2_02_ficha.json"]
faltan = [p.name for p in necesarios if not p.exists()]
if faltan:
    raise FileNotFoundError(f"Faltan productos en {RES}: {faltan}. Ejecute IA2_01 e IA2_02 con la provincia {PROVINCIA}.")
CLAVE = pd.read_csv(necesarios[0])
META1 = json.loads(necesarios[1].read_text(encoding="utf-8"))
RIESGO = pd.read_csv(necesarios[2])
FICHA = json.loads(necesarios[3].read_text(encoding="utf-8"))
PRINC = CLAVE.loc[CLAVE.principal, "categoria"].unique().tolist()
AMEN_REL = FICHA["amenazas_relevantes"]
FACT = FICHA["factores_criticos"]
print(f"Inventario {META1['anio']}: total {META1['total_tCO2e']/1000:,.0f} kt CO2-eq (± {META1['incertidumbre_pct']:.0f} %)".replace(",", " "))
print("Categorías principales:", "; ".join(PRINC))
print(f"Sitio: {FICHA['sitio']['nombre']} · amenazas de riesgo medio o mayor: {[RG.AMENAZAS[a] for a in AMEN_REL] or 'ninguna'}")
print("Indicadores críticos de vulnerabilidad:", FACT or "ninguno")
NE = [k for k, v in META1["completitud"].items() if v.startswith("NE")]
if NE:
    print("Categorías no estimadas en el inventario (NE):", NE, "→ sus opciones no se sugieren; puede agregarlas como propias.")

## 3. Opciones sugeridas por sus resultados

In [ ]:
SUG = ES.sugerir(PRINC, AMEN_REL, FACT)
SUG["n_motivos"] = SUG.motivo.str.count(";") + 1
SUG = SUG.sort_values(["n_motivos", "tipo"], ascending=[False, True])
pd.set_option("display.max_colwidth", 90)
display(SUG[["opcion", "tipo", "enfoque", "motivo", "sector"]])
print("Tipo: M = mitigación, A = adaptación, MA = las dos.  Catálogo completo: ES.catalogo()")
print(f"\n{len(SUG)} opciones vinculadas con sus resultados; abajo, las 12 con más motivos. Puede cambiar o agregar ids del catálogo.")
print("# ---- Copie a la celda 6 las líneas de las opciones que elija (entre 6 y 12; al menos 3 M o MA y 3 A o MA) ----")
print("# id; efectividad; costo; factibilidad; cobeneficios; sinergia; mala_adaptacion; alineacion; justificación")
for i, r in SUG.head(12).iterrows():
    print(f"{i}; 3; 3; 3; 3; {4 if r.tipo == 'MA' else 2}; 3; 3; {r.opcion[:60]}: justifique aquí")

## 4. Criterios de evaluación (1–5; en todos, 5 es lo mejor)

| Criterio | Pregunta | 1 | 5 |
|---|---|---|---|
| efectividad | ¿Cuánto reduce las emisiones de su inventario o el riesgo del sitio? | casi nada | mucho, en una categoría principal o en la amenaza principal |
| costo | ¿Cuánto cuesta implementarla y mantenerla? | muy cara | barata |
| factibilidad | ¿Es viable técnica, institucional y socialmente en su provincia? | no hay capacidades ni actores | hay actores, normas y experiencia |
| cobeneficios | ¿Trae otros beneficios (salud, empleo, agua, biodiversidad)? | ninguno | muchos y claros |
| sinergia | ¿Sirve a la vez para mitigar y para adaptarse? | solo una, y con conflicto | las dos, claramente |
| mala_adaptacion | ¿Puede trasladar o aumentar el riesgo, o la vulnerabilidad de otros? | riesgo alto | riesgo bajo |
| alineacion | ¿Está en la NDC, el PLANACC o el PDOT? | no aparece | aparece como prioridad |

In [ ]:
# @title 5. Opciones propias (opcional) y pesos de los criterios
# Una por línea: id que empiece con P; nombre; tipo (M, A o MA)
OPCIONES_PROPIAS = """
"""
W_EFECTIVIDAD = 0.25  # @param {type:"slider", min:0, max:0.5, step:0.05}
W_COSTO = 0.10  # @param {type:"slider", min:0, max:0.5, step:0.05}
W_FACTIBILIDAD = 0.20  # @param {type:"slider", min:0, max:0.5, step:0.05}
W_COBENEFICIOS = 0.15  # @param {type:"slider", min:0, max:0.5, step:0.05}
W_SINERGIA = 0.10  # @param {type:"slider", min:0, max:0.5, step:0.05}
W_MALA_ADAPTACION = 0.10  # @param {type:"slider", min:0, max:0.5, step:0.05}
W_ALINEACION = 0.10  # @param {type:"slider", min:0, max:0.5, step:0.05}
JUSTIFICACION_PESOS = "Por qué pesa más la efectividad y la factibilidad en mi provincia."  # @param {type:"string"}

In [ ]:
# @title 6. Evaluación (pegue aquí el bloque de la celda 3 y edítelo)
EVALUACION = """
"""

## 7. Matriz, ranking y robustez

In [ ]:
CAT = ES.catalogo()
for linea in OPCIONES_PROPIAS.strip().splitlines():
    p = [x.strip() for x in linea.split(";")]
    if len(p) >= 3 and p[0].upper().startswith("P"):
        CAT.loc[p[0]] = [p[1], p[2].upper(), "propia", [], [], [], "—", ""]
W = {"efectividad": W_EFECTIVIDAD, "costo": W_COSTO, "factibilidad": W_FACTIBILIDAD, "cobeneficios": W_COBENEFICIOS,
     "sinergia": W_SINERGIA, "mala_adaptacion": W_MALA_ADAPTACION, "alineacion": W_ALINEACION}
sw = sum(W.values())
W = {k: v / sw for k, v in W.items()}
if abs(sw - 1) > 0.01:
    print(f"Los pesos sumaban {sw:.2f}; se normalizaron a 1.")
POR_DEFECTO = not EVALUACION.strip()
if POR_DEFECTO:
    print("⚠ Celda 6 vacía: se evalúan todas las sugerencias con 3. NO es válido para entregar.")
    EV = pd.DataFrame({k: 3 for k in ES.CRITERIOS}, index=SUG.index[:10]); EV["justificacion"] = ""
else:
    EV = ES.leer_evaluacion(EVALUACION)
desconocidas = [i for i in EV.index if i not in CAT.index]
if desconocidas:
    raise ValueError(f"Ids que no están en el catálogo ni en sus opciones propias: {desconocidas}")
EV = CAT.loc[EV.index, ["opcion", "tipo", "enfoque", "sector", "alerta_mala_adaptacion"]].join(EV)
EV["puntaje"] = ES.puntuar(EV, W).round(2)
EV["puntaje_pesos_iguales"] = ES.puntuar(EV, {k: 1 / len(ES.CRITERIOS) for k in ES.CRITERIOS}).round(2)
EV = EV.sort_values("puntaje", ascending=False)
EV["rango"] = range(1, len(EV) + 1)
EV["rango_pesos_iguales"] = EV.puntaje_pesos_iguales.rank(ascending=False, method="min").astype(int)
display(EV[["opcion", "tipo", *ES.CRITERIOS, "puntaje", "rango", "rango_pesos_iguales"]])
top = set(EV.index[:5]); top_eq = set(EV.sort_values("puntaje_pesos_iguales", ascending=False).index[:5])
ROBUSTEZ = len(top & top_eq) / max(1, min(5, len(EV)))
print(f"Robustez: {len(top & top_eq)} de las 5 primeras se mantienen con pesos iguales ({100*ROBUSTEZ:.0f} %).")

avisos = []
n_m = EV.tipo.isin(["M", "MA"]).sum(); n_a = EV.tipo.isin(["A", "MA"]).sum()
if not 6 <= len(EV) <= 12: avisos.append(f"Evaluó {len(EV)} opciones; se piden entre 6 y 12.")
if n_m < 3: avisos.append(f"Solo {n_m} opciones de mitigación (M o MA); se piden al menos 3.")
if n_a < 3: avisos.append(f"Solo {n_a} opciones de adaptación (A o MA); se piden al menos 3.")
cod_ev = {c for i in EV.index for c in CAT.loc[i, "categorias"]}
for c in PRINC[:3]:  # las tres categorías que más emiten
    if c.split(" ")[0] not in cod_ev and not c.startswith("Memo"):
        avisos.append(f"La categoría «{c}», una de las tres que más emiten, no tiene ninguna opción evaluada.")
am_ev = {a for i in EV.index for a in CAT.loc[i, "amenazas"]}
for a in AMEN_REL:
    if a not in am_ev: avisos.append(f"La amenaza «{RG.AMENAZAS[a]}» del sitio no tiene ninguna opción de adaptación evaluada.")
sin_j = [i for i, r in EV.iterrows() if not str(r.justificacion).strip() or "justifique aquí" in str(r.justificacion)]
if sin_j: avisos.append(f"Falta la justificación de: {', '.join(sin_j)}.")
for i, r in EV.iterrows():
    if r.alerta_mala_adaptacion and r.mala_adaptacion >= 4:
        avisos.append(f"{i}: el catálogo advierte «{r.alerta_mala_adaptacion}»; explique por qué calificó riesgo bajo de mala adaptación.")
if JUSTIFICACION_PESOS.startswith("Por qué pesa"): avisos.append("Falta la justificación de los pesos.")
print("\n" + ("Sin observaciones." if not avisos else "Revise antes de entregar:\n- " + "\n- ".join(avisos)))

## 8. Figuras de la matriz

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(17, 0.55 * len(EV) + 3), gridspec_kw={"width_ratios": [1.5, 1], "wspace": 0.35})
ax = axs[0]
crit = list(ES.CRITERIOS)
im = ax.imshow(EV[crit].values, cmap="YlGn", vmin=1, vmax=5, aspect="auto")
ax.set_xticks(range(len(crit))); ax.set_xticklabels([c.replace("_", " ") for c in crit], rotation=30, ha="right")
etq = [f"{i} {o[:42]}{'…' if len(o) > 42 else ''} ({t})" for i, o, t in zip(EV.index, EV.opcion, EV.tipo)]
ax.set_yticks(range(len(EV))); ax.set_yticklabels(etq, fontsize=9.5)
for y in range(len(EV)):
    for x in range(len(crit)):
        v = int(EV[crit].iloc[y, x]); ax.text(x, y, v, ha="center", va="center", fontsize=10, color="white" if v >= 5 else "#222222")
ax.set_title("a) Matriz de evaluación (1–5) y puntaje ponderado", loc="left", fontweight="bold")
for y, v in enumerate(EV.puntaje): ax.text(len(crit) - 0.3, y, f"{v:.2f}", va="center", fontsize=10.5, fontweight="bold")
ax = axs[1]
COLT = {"M": "#C2531F", "A": "#2F6DB5", "MA": "#6A994E"}
ax.scatter(EV.factibilidad, EV.efectividad, s=60 * EV.cobeneficios, c=[COLT.get(t, "#888888") for t in EV.tipo], alpha=0.75, edgecolor="white")
for (fx, ef), g in EV.groupby(["factibilidad", "efectividad"]):
    ax.text(fx, ef + 0.16, "\n".join(g.index), ha="center", va="bottom", fontsize=10, linespacing=1.1)
ax.axhline(3, color="#AAAAAA", lw=0.8, ls="--"); ax.axvline(3, color="#AAAAAA", lw=0.8, ls="--")
ax.set_xlim(0.5, 5.5); ax.set_ylim(0.5, 5.5); ax.set_xlabel("factibilidad"); ax.set_ylabel("efectividad")
ax.text(5.4, 5.4, "priorizar", ha="right", va="top", color="#4E7A38", fontweight="bold")
ax.text(0.6, 5.4, "preparar condiciones", va="top", color="#6A7179"); ax.text(5.4, 0.6, "ganancias rápidas, pequeñas", ha="right", color="#6A7179")
for t, c in COLT.items(): ax.scatter([], [], c=c, s=80, label={"M": "mitigación", "A": "adaptación", "MA": "las dos"}[t])
ax.legend(loc="lower left", frameon=False, title="tamaño = cobeneficios")
ax.set_title("b) Efectividad frente a factibilidad", loc="left", fontweight="bold")
fig.suptitle(f"Matriz de estrategias · {PROVINCIA} · sitio {FICHA['sitio']['nombre']}", x=0.01, ha="left", fontsize=15, fontweight="bold")
fig.savefig(RES / f"IA2_03_matriz_{SP}.png", dpi=200, bbox_inches="tight", facecolor="white")
plt.show()

## 9. Cartera priorizada y archivos para el capítulo 2

In [ ]:
# Las mejores opciones por puntaje, con al menos 2 de mitigación y 2 de adaptación entre las 6 primeras
cartera, m, a = [], 0, 0
for i, r in EV.iterrows():
    if len(cartera) >= 6: break
    faltan_m, faltan_a = max(0, 2 - m), max(0, 2 - a)
    libres = 6 - len(cartera)
    es_m, es_a = r.tipo in ("M", "MA"), r.tipo in ("A", "MA")
    if libres <= faltan_m + faltan_a and not ((es_m and faltan_m) or (es_a and faltan_a)):
        continue
    cartera.append(i); m += es_m; a += es_a
CARTERA = EV.loc[cartera, ["opcion", "tipo", "enfoque", "puntaje", "sector", "alerta_mala_adaptacion", "justificacion"]]
CARTERA["responde_a"] = [SUG.motivo.get(i, "opción propia") for i in cartera]
display(CARTERA)
EV.to_csv(RES / f"IA2_03_evaluacion_{SP}.csv", encoding="utf-8")
CARTERA.to_csv(RES / f"IA2_03_cartera_{SP}.csv", encoding="utf-8")
meta = {"material": "CC-IAM-P2-03 v1.0", "estudiante": ESTUDIANTE, "provincia": PROVINCIA, "pesos": W,
        "justificacion_pesos": JUSTIFICACION_PESOS, "robustez_top5": ROBUSTEZ, "evaluacion_por_defecto": POR_DEFECTO,
        "categorias_principales": PRINC, "amenazas_relevantes": AMEN_REL, "factores_criticos": FACT,
        "cartera": cartera, "avisos": avisos, "fecha": datetime.date.today().isoformat()}
(RES / "IA2_03_metadatos.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
print(sorted(p.name for p in RES.glob("IA2_*")))

## 10. Preguntas para el capítulo 2

1. ¿Qué opciones quedaron en la cartera y a qué responden (categoría del inventario, amenaza o indicador de vulnerabilidad del sitio)?
2. ¿Hay opciones con sinergia mitigación–adaptación? ¿Cuáles y por qué se refuerzan?
3. ¿Su ranking es robusto? Si cambia con pesos iguales, ¿qué criterio explica el cambio y cómo justifica sus pesos?
4. ¿Qué opción tiene más riesgo de mala adaptación y cómo lo evitaría (diseño, lugar, a quién beneficia)?
5. ¿Qué actores de su provincia (GAD, MAATE, MAG, juntas de agua, comunidades) tendrían que ejecutar la primera opción de la cartera? ¿Qué la hace factible o la frena?
6. ¿Cómo se alinea la cartera con la Segunda NDC y con el PLANACC? ¿Qué falta en esos planes para su provincia?

**Para el informe:** las opciones evaluadas con su justificación, los pesos y su justificación, la prueba de robustez y la cartera final.

**Citas.** IPCC (2022). *Climate Change 2022: Mitigation of Climate Change*, Grupo de Trabajo III, caps. 7, 10 y 12 · IPCC (2022). *Climate Change 2022: Impacts, Adaptation and Vulnerability*, Grupo de Trabajo II, caps. 12 y 17 · MAATE (2025). *Segunda Contribución Determinada a Nivel Nacional del Ecuador* · MAATE. *Plan Nacional de Adaptación al Cambio Climático (PLANACC)* · Belton, V. y Stewart, T. (2002). *Multiple Criteria Decision Analysis*. Springer.